# Prompting Techniques — Hands-On

**Session 7 — Applied Natural Language Processing**

This notebook explores the core prompting techniques covered in today's lecture. You will experiment with zero-shot, one-shot, and few-shot prompting, chain-of-thought reasoning, role prompting, and temperature effects — all using **sentiment analysis** as the running example.

We use **GitHub Models** for API access (free with a GitHub account) or local ollama model (gpt-oss or llama3.2). Full API coverage is Session 8. 

**Prerequisites:** Set your GitHub token as an environment variable before running, you should have it set in the `.env` file or export it in your terminal:
```
export GITHUB_TOKEN=your_token_here
```

In [25]:
import os
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI

# GitHub Models client — free with a GitHub account
# Full API coverage in Session 8; this is a lightweight preview
# client = OpenAI(
#     base_url="https://models.inference.ai.azure.com",
#     api_key=os.environ.get("GITHUB_TOKEN", "set-your-github-token"),
# )
# MODEL = "gpt-4o-mini"

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama-token",
)

MODEL = "llama3.2"

def chat(prompt, system=None, temperature=0.3, max_tokens=300):
    """Simple helper to send a prompt and get a response."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print(chat("Confirm that the API is working by responding with 'Ready.'"))

print("Ready.")


---

## Part 1 — Zero-Shot, One-Shot, and Few-Shot Prompting

Prompting techniques differ in how many examples (demonstrations) you provide before asking the model to perform the task.

| Technique | Examples provided | When to use |
|---|---|---|
| Zero-shot | None | General tasks, quick tests |
| One-shot | 1 per class | When you have one clear example |
| Few-shot | 3+ examples | Consistent format, domain-specific style |

We'll test all three on the same sentiment analysis input.

In [26]:
review = "Despite the long wait, the food was absolutely delicious and the staff were incredibly friendly."

# Zero-shot: direct instruction, no examples
zero_shot_prompt = f"""Analyze the sentiment of the following text.
Classify it as Positive, Negative, or Neutral.

Text: "{review}"
Sentiment:"""

result = chat(zero_shot_prompt)
print("Zero-shot result:")
print(result)

Zero-shot result:
The sentiment of the text is Positive. The use of words such as "absolutely" and "incredibly" convey a strong sense of enthusiasm and satisfaction with the experience. Additionally, the mention of both the quality of the food and the friendliness of the staff suggests that the reviewer had a very positive overall experience.


In [27]:
# One-shot: one example before the target
one_shot_prompt = f"""Analyze sentiment. Classify as Positive, Negative, or Neutral.

Example:
Text: "The movie was a complete waste of time. The plot was confusing."
Sentiment: Negative

Now classify:
Text: "{review}"
Sentiment:"""

result = chat(one_shot_prompt)
print("One-shot result:")
print(result)

One-shot result:
Based on the text:

"Despite the long wait, the food was absolutely delicious and the staff were incredibly friendly."

I would classify the sentiment as Positive. The use of positive adjectives such as "absolutely delicious" and "incredibly friendly" indicates a very favorable tone, outweighing any potential negative aspect (the long wait).


In [28]:
# Few-shot: multiple examples establish the pattern clearly
few_shot_prompt = f"""Classify the sentiment of each text as Positive, Negative, or Neutral.

Text: "The new smartphone exceeded all my expectations. Camera quality is outstanding."
Sentiment: Positive

Text: "I'm so disappointed with the service. My order arrived damaged."
Sentiment: Negative

Text: "The package arrived on time. Nothing special, but it works."
Sentiment: Neutral

Text: "{review}"
Sentiment:"""

result = chat(few_shot_prompt)
print("Few-shot result:")
print(result)

Few-shot result:
Positive.

The text mentions that the food was "absolutely delicious" and the staff were "incredibly friendly", indicating a very positive experience.


**What to notice:**
- All three likely give the correct answer here — this review is unambiguous.
- Try a more ambiguous review (e.g. "The price was high but the quality justified it") and compare how each technique handles it.
- Few-shot tends to be most consistent in format — the model learns from the examples exactly how to format the output.

---

## Part 2 — Chain-of-Thought Prompting

Standard prompting asks for the answer directly. **Chain-of-thought (CoT)** prompting asks the model to reason step-by-step before answering — this significantly improves performance on complex reasoning tasks.

Two variants:
- **Standard CoT**: Provide a worked example with reasoning steps
- **Zero-shot CoT**: Add "Let's think step by step" (Kojima et al., 2022) — no examples needed

In [29]:
# A tricky review with mixed sentiment
tricky_review = "Despite the long wait and slightly overcooked pasta, the atmosphere was wonderful and the dessert was the best I've ever had."

# Standard CoT: explicit reasoning steps in the instruction
cot_prompt = f"""Analyze the sentiment of the following text by following these steps:

1. Identify key emotional words or phrases
2. Determine if each is positive, negative, or neutral
3. Consider any context that modifies their meaning
4. Conclude with the overall sentiment

Text: "{tricky_review}"

Analysis:"""

result = chat(cot_prompt, temperature=0.1, max_tokens=400)
print("Chain-of-thought result:")
print(result)

Chain-of-thought result:
1. Key emotional words or phrases:
- "long wait"
- "slightly overcooked pasta" (negative)
- "wonderful" (positive)
- "best I've ever had" (strongly positive)

2. Determining sentiment:
- "long wait" is neutral, as it's a factual statement about the experience.
- "slightly overcooked pasta" is negative, indicating a disappointment with the food quality.
- "wonderful" is positive, describing the atmosphere.
- "best I've ever had" is strongly positive, expressing enthusiasm and high praise for the dessert.

3. Considering context:
The text acknowledges a negative aspect (overcooked pasta) but frames it as a minor issue compared to the overall positive experience. The use of "slightly overcooked" also suggests that it was not a catastrophic mistake, which might temper the negative sentiment slightly.

4. Concluding with the overall sentiment:
Despite the negative comment about the pasta, the text concludes on a strongly positive note, highlighting the wonderful atm

In [30]:
# Zero-shot CoT: just add "Let's think step by step"
zero_cot_prompt = f"""Analyze the sentiment of this text. Let's think step by step.

Text: "{tricky_review}"
"""

result = chat(zero_cot_prompt, temperature=0.1, max_tokens=400)
print("Zero-shot CoT result:")
print(result)

Zero-shot CoT result:
Let's analyze the sentiment of the text step by step:

1. The first phrase, "Despite the long wait," sets a somewhat negative tone. The word "despite" implies that there were some drawbacks to the experience.

2. The next phrase, "and slightly overcooked pasta," reinforces this idea. The use of the word "slightly" suggests that it's not a major issue, but still, it's an inconvenience.

3. However, the third phrase, "the atmosphere was wonderful," shifts the tone significantly. The word "wonderful" is a strong positive adjective that indicates a very pleasant and enjoyable experience.

4. Finally, the last sentence, "and the dessert was the best I've ever had," further reinforces the positive sentiment. The use of the superlative form "best" emphasizes just how exceptional the dessert was.

Overall, despite some minor drawbacks (the wait and slightly overcooked pasta), the text expresses a predominantly positive sentiment. The reviewer seems to have enjoyed their e

**What to notice:**
- CoT produces a more nuanced response — it surfaces the tension between positive and negative elements.
- The simple few-shot prompt from Part 1 might collapse mixed sentiment into one label. CoT surfaces the nuance.
- Zero-shot CoT ("Let's think step by step") is surprisingly effective for its simplicity.

---

## Part 3 — Role Prompting

Assigning the model a role or persona can improve the quality and consistency of its responses. The model draws on its training to inhabit the role.

In [31]:
review = "The product arrived late, but the quality exceeded my expectations."

# Without role
no_role = chat(f"Analyze the sentiment: '{review}'")

# With expert role
with_role = chat(
    f"Analyze the sentiment: '{review}'",
    system="You are an expert in sentiment analysis with 10 years of experience. "
           "Provide a brief, structured analysis covering: overall sentiment, key emotional signals, "
           "and any nuance or ambiguity in the text."
)

print("Without role:")
print(no_role)
print()
print("With sentiment expert role:")
print(with_role)

Without role:
The sentiment of this statement can be analyzed as follows:

1. The initial phrase "The product arrived late" has a negative connotation, implying disappointment or frustration with the delay.
2. However, the second part of the sentence "but the quality exceeded my expectations" shifts the tone to one of surprise and delight. The use of the word "exceeded" suggests that the speaker was pleasantly surprised by the quality of the product.

Overall, the sentiment of this statement is mixed, but leaning towards being positive. The speaker acknowledges a negative experience (the delay), but frames it as a minor issue compared to their overall satisfaction with the product's quality.

The sentiment can be broken down into:

* Negative: Delay (frustration)
* Neutral/Mixed: Product arrival time
* Positive: Quality exceeded expectations (surprise and delight)

The tone is generally neutral, with a hint of positivity. The speaker seems to be focusing on the positive aspect of their

**What to notice:**
- Role prompting tends to produce more structured, domain-aware responses.
- It's particularly useful when you want a consistent format across many outputs.
- The role sets implicit expectations — an "expert" produces more detailed analysis than a generic response.

---

## Part 4 — Temperature Effects

**Temperature** controls the randomness of the model's output.

| Range | Behaviour | Good for |
|---|---|---|
| 0.0 – 0.3 | Deterministic, consistent | Classification, fact extraction, evaluation |
| 0.4 – 0.7 | Balanced | General tasks |
| 0.8 – 1.0 | Creative, diverse | Brainstorming, creative writing |

Note: temperature is an **inference parameter**, not a model hyperparameter — it doesn't change the model, it changes how the model samples from its output distribution.

In [32]:
prompt = "Write a one-sentence product tagline for a reusable coffee cup."

for temp in [0.0, 0.5, 1.0]:
    results = []
    for _ in range(3):  # run 3 times to show consistency/variation
        result = chat(prompt, temperature=temp, max_tokens=100)
        results.append(result)

    print(f"Temperature = {temp}:")
    for i, r in enumerate(results, 1):
        print(f"  Run {i}: {r}")
    print()

Temperature = 0.0:
  Run 1: "Fuel your daily grind with a cup that's always on the right brew."
  Run 2: "Fuel your daily grind with a cup that's always on the right brew."
  Run 3: "Fuel your daily grind with a cup that's always on the right brew."

Temperature = 0.5:
  Run 1: "Fuel your day, not the waste."
  Run 2: "Fuel your daily grind with a conscience, our reusable cups keep you caffeinated and the planet happy."
  Run 3: "Fuel your daily grind with a sip, not waste."

Temperature = 1.0:
  Run 1: "Warm up to sustainability with every sip, and keep the planet brewing fresh."
  Run 2: "Fuel your daily grind with a second chance, every time, with our durable and eco-friendly reusable coffee cup."
  Run 3: "Pour yourself a sustainable future, one cup at a time with [Brand Name]'s eco-friendly reusable coffee cups."



**What to notice:**
- At `temperature=0.0`: all 3 runs should produce the same (or very similar) output.
- At `temperature=1.0`: each run produces a noticeably different tagline.
- For sentiment classification, use low temperature — you want consistent labels.
- For brainstorming or creative writing, use higher temperature to get diverse outputs.

---

## Part 5 — Grammar Correction Bot (Worked Example)

A well-structured prompt uses multiple elements together: a **role**, a **task**, detailed **instructions**, and **constraints**. This example from the lecture shows a Grammar Correction Bot prompt.

This is the kind of multi-element prompt you'd build for a production use case.

In [33]:
grammar_bot_system = """Role: You are an expert English language teacher with extensive experience in grammar correction and explanation.

Task: Correct the grammar in the given text and provide explanations for the corrections.

Instructions: Follow these steps when correcting grammar:
1. Read the entire text to understand the context
2. Identify and correct all grammatical errors
3. Explain each correction clearly and concisely
4. Preserve the original meaning

Additional guidelines:
- If a sentence is grammatically correct, state "No correction needed" and briefly explain why.
- Pay attention to: subject-verb agreement, verb tenses, pronoun usage, word order, articles and prepositions.
- Format your response as:
  Original: [original sentence]
  Corrected: [corrected sentence]
  Explanation: [what was changed and why]"""

test_sentences = [
    "She don't know what are she doing yesterday at the park.",
    "The team have been working very hard on their project since last week.",
    "Yesterday I go to the store and buyed some groceries.",
]

for sentence in test_sentences:
    print(f"Input: {sentence}")
    print()
    result = chat(sentence, system=grammar_bot_system, temperature=0.1, max_tokens=300)
    print(result)
    print("-" * 60)
    print()

Input: She don't know what are she doing yesterday at the park.

Original: She don't know what are she doing yesterday at the park.
Corrected: She didn't know what she was doing yesterday at the park.

Explanation:
1. "don't" is a contraction of "do not", whereas "didn't" is the correct form to use with the verb "know". In this context, we need the negative form of the verb "to know".
2. The apostrophe in "don't" is used for contractions, but here it's incorrect because we're using a contraction that doesn't require an apostrophe ("didn't"). Instead, we should use the correct spelling of the word.
3. "are" is a plural verb form, whereas "was" is the correct singular form to use with the subject "she". The verb tense also needs to match; in this case, the past simple tense "was" is used because it's describing an action that happened at a specific point in the past.
4. Finally, we need to add an article ("the") before the noun "park", as it's a specific location being referred to.
-----

**What to notice:**
- The system prompt does a lot of heavy lifting — the user message is just the raw text.
- Structured instructions (numbered steps + explicit output format) produce consistent, structured output.
- This pattern — detailed system prompt, minimal user input — is the foundation of production LLM applications.

---

## Part 6 — LLM Challenges: Live Demos

The lecture covered several LLM limitations. Let's observe them directly.

In [34]:
# Hallucination: ask for something obscure/nonexistent
# LLMs can confidently produce plausible-sounding but false information

hallucination_prompt = """List 3 peer-reviewed papers published in 2019-2020 specifically about
using BERT for sentiment analysis in the restaurant industry,
including the journal name, volume, and DOI."""

result = chat(hallucination_prompt, temperature=0.3, max_tokens=400)
print("Model response:")
print(result)
print()
print("NOTE: Verify each of these references — LLMs frequently fabricate citations.")
print("This is known as hallucination: confident, plausible, but potentially false.")

Model response:
I was unable to find any specific peer-reviewed papers published in 2019-2020 that are focused on using BERT for sentiment analysis in the restaurant industry.

However, I can provide some general information about BERT-based sentiment analysis in the restaurant industry. BERT (Bidirectional Encoder Representations from Transformers) is a popular pre-trained language model that has been widely used for various natural language processing tasks, including sentiment analysis.

There are several papers that have explored the use of BERT for sentiment analysis in the restaurant industry, but they may not be specifically focused on this topic. Some examples include:

1. "BERT for Sentiment Analysis: A Review" by S. M. Hasan et al., published in the Journal of Intelligent Information Systems, Volume 56, Issue 2 (2020), DOI: 10.1007/s10844-019-05217-9.

This paper provides a review of various BERT-based approaches for sentiment analysis and discusses their applications in diff

In [35]:
# Sycophancy: LLMs tend to agree with confident false claims

correct_claim = "The Eiffel Tower is located in Paris, France."
incorrect_claim = "The Eiffel Tower is located in London, England, right? I'm pretty sure about this."

for claim in [correct_claim, incorrect_claim]:
    result = chat(f"Is this correct? {claim}", temperature=0.1, max_tokens=150)
    print(f"Claim: {claim}")
    print(f"Response: {result}")
    print()

Claim: The Eiffel Tower is located in Paris, France.
Response: Yes, that's absolutely correct! The Eiffel Tower is indeed a famous landmark located in Paris, France. It was built for the 1889 World's Fair and has since become one of the most iconic symbols of Paris and France.

Claim: The Eiffel Tower is located in London, England, right? I'm pretty sure about this.
Response: I think there may be a mistake here! The Eiffel Tower is actually located in Paris, France, not London, England. It's one of the most iconic landmarks in Paris and was built for the 1889 World's Fair. London has its own famous landmarks like Buckingham Palace, the Tower of London, and Big Ben (now officially known as the Elizabeth Tower), but the Eiffel Tower is not one of them.



In [36]:
# Non-determinism: same prompt, different outputs at higher temperature

prompt = "In one sentence, what is the most important thing to know about prompt engineering?"

print("Running the same prompt 5 times at temperature=0.8:")
print()
for i in range(5):
    result = chat(prompt, temperature=0.8, max_tokens=80)
    print(f"Run {i+1}: {result}")

Running the same prompt 5 times at temperature=0.8:

Run 1: The most important thing to know about prompt engineering is that it requires a delicate balance between providing enough context and specificity for the AI model to understand, while also allowing for flexibility and creativity in generating desired outputs.
Run 2: The most important thing to know about prompt engineering is that it's all about crafting and fine-tuning input prompts to elicit specific, accurate, and relevant responses from AI models, requiring a deep understanding of language nuances, context, and model capabilities.
Run 3: The most important thing to know about prompt engineering is that it requires a deep understanding of the underlying language model's capabilities, limitations, and biases in order to craft high-quality prompts that elicit accurate and relevant responses.
Run 4: The most important thing to know about prompt engineering is that it requires a deep understanding of how language models process

---

## Summary

| Technique | When to use | Key parameter |
|---|---|---|
| Zero-shot | Quick tests, general tasks | None |
| One-shot | When you have one clear example | 1 example |
| Few-shot | Consistent format, domain tasks | 3+ examples |
| Chain-of-thought | Complex reasoning, nuanced tasks | Reasoning steps |
| Role prompting | Structured output, domain expertise | System prompt |
| Low temperature | Classification, evaluation, consistency | `temperature=0.0` |
| High temperature | Brainstorming, creative writing | `temperature=0.8+` |

**Key takeaway:** Prompting is iterative. Start simple (zero-shot), add examples (few-shot) if needed, add reasoning (CoT) for complex tasks, and use roles to shape output format and quality.

**Next:** `02_llm_evaluation.ipynb` — how do we evaluate whether these prompts actually work?